In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# The URL points directly to the raw CSV file on GitHub for Confirmed Global Cases
URL = 'https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/csse_covid_19_data/csse_covid_19_time_series/time_series_covid19_confirmed_global.csv'
df_raw = pd.read_csv(URL)

# Inspect the first few rows to confirm the data loaded correctly
print("Raw Data Head:")
print(df_raw.head(5))

Raw Data Head:
  Province/State Country/Region       Lat       Long  1/22/20  1/23/20  \
0            NaN    Afghanistan  33.93911  67.709953        0        0   
1            NaN        Albania  41.15330  20.168300        0        0   
2            NaN        Algeria  28.03390   1.659600        0        0   
3            NaN        Andorra  42.50630   1.521800        0        0   
4            NaN         Angola -11.20270  17.873900        0        0   

   1/24/20  1/25/20  1/26/20  1/27/20  ...  2/28/23  3/1/23  3/2/23  3/3/23  \
0        0        0        0        0  ...   209322  209340  209358  209362   
1        0        0        0        0  ...   334391  334408  334408  334427   
2        0        0        0        0  ...   271441  271448  271463  271469   
3        0        0        0        0  ...    47866   47875   47875   47875   
4        0        0        0        0  ...   105255  105277  105277  105277   

   3/4/23  3/5/23  3/6/23  3/7/23  3/8/23  3/9/23  
0  209369  20

In [8]:
# The URL for the global deaths time series
DEATHS_URL = 'https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/csse_covid_19_data/csse_covid_19_time_series/time_series_covid19_deaths_global.csv'
df_deaths_raw = pd.read_csv(DEATHS_URL)

print("Raw Deaths Data Head:")
print(df_deaths_raw.head(3))

Raw Deaths Data Head:
  Province/State Country/Region       Lat       Long  1/22/20  1/23/20  \
0            NaN    Afghanistan  33.93911  67.709953        0        0   
1            NaN        Albania  41.15330  20.168300        0        0   
2            NaN        Algeria  28.03390   1.659600        0        0   

   1/24/20  1/25/20  1/26/20  1/27/20  ...  2/28/23  3/1/23  3/2/23  3/3/23  \
0        0        0        0        0  ...     7896    7896    7896    7896   
1        0        0        0        0  ...     3598    3598    3598    3598   
2        0        0        0        0  ...     6881    6881    6881    6881   

   3/4/23  3/5/23  3/6/23  3/7/23  3/8/23  3/9/23  
0    7896    7896    7896    7896    7896    7896  
1    3598    3598    3598    3598    3598    3598  
2    6881    6881    6881    6881    6881    6881  

[3 rows x 1147 columns]


In [9]:
# --- CLEANING DEATHS ---
# 1. Sum by country and drop location columns
df_deaths_agg = df_deaths_raw.drop(columns=['Lat', 'Long', 'Province/State']).groupby('Country/Region').sum()
# 2. Transpose (flip) and convert index to dates
df_deaths_ts = df_deaths_agg.T
df_deaths_ts.index = pd.to_datetime(df_deaths_ts.index)

# --- CLEANING CASES ---
df_cases_agg = df_raw.drop(columns=['Lat', 'Long', 'Province/State']).groupby('Country/Region').sum()
# 4. Transpose and convert index to dates
df_cases_ts = df_cases_agg.T
df_cases_ts.index = pd.to_datetime(df_cases_ts.index)

print("Data cleaning complete. Tables created for both Deaths and Cases.")

Data cleaning complete. Tables created for both Deaths and Cases.


C:\Users\zsche\AppData\Local\Temp\ipykernel_26828\2401683264.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_deaths_ts.index = pd.to_datetime(df_deaths_ts.index)
C:\Users\zsche\AppData\Local\Temp\ipykernel_26828\2401683264.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_cases_ts.index = pd.to_datetime(df_cases_ts.index)


In [18]:
# 1. Get the most recent date's totals for all countries
# We look at the last row (iloc[-1]) of our time series data
latest_totals = df_deaths_ts.iloc[-1].sort_values(ascending=False)

#2. Now find the top 10 countries with most deaths
# .head(10) picks the top ones, .index grabs the names, .tolist() makes it a list
top_10_countries = latest_totals.head(10).index.tolist()

# 3. Isolate the US data into its own DataFrame for plotting later
# Note: In this dataset, the USA is labeled as 'US'
usa_cases = df_cases_ts['US']
usa_deaths = df_deaths_ts['US']
# 3. Calculate Case Fatality Rate (CFR) - Ratio of Deaths to Case
usa_case_to_death_ratio = (usa_deaths/usa_cases) * 100

print(f"Top 10 countries with most cases are:\n {top_10_countries}")
print("-"*290)
print(f"USA Total Cases: {usa_cases.iloc[-1]:,.0f}")
print(f"USA Total Deaths: {usa_deaths.iloc[-1]:,.0f}")
print(f"USA Case Fatality Rate: {usa_case_to_death_ratio.iloc[-1]:.2f}%")



Top 10 countries with most cases are:
 ['US', 'Brazil', 'India', 'Russia', 'Mexico', 'United Kingdom', 'Peru', 'Italy', 'Germany', 'France']
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
USA Total Cases: 103,802,702
USA Total Deaths: 1,123,836
USA Case Fatality Rate: 1.08%
